[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/03_cluster.ipynb)

# Shared topic space — soft MPPCA, harden views, lenses

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: Once phrases exist, we place them in one embedding space. Soft-throughout
Address (CIP-0017) fits a **mixture of probabilistic PCAs**: $k$-means only
initialises component means; soft responsibilities $r_{ik}$ are the primary
topic state. Hard labels and exemplars are a *late view* for LLM labelling and
lens prompts. This notebook teaches that split and shows a PCA view as a
diagnostic — not as the generative story for the phrases.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

Hard $k$-means gives every phrase a single topic label. Soft-throughout instead
keeps a distribution over topics. Multilogue's [`mppca_fit`](https://mlatcl.github.io/multilogue/api/mppca/)
uses $k$-means only to seed means, then EM under a **fixed** topic noise
$\sigma^2_{\mathrm{topic}}$ (independent of the lens-level $\sigma^2$ in
notebook 08).

Labels name topics; lenses group topics into coarser framings. Both LLM steps
consume a **harden view** (MAP labels / top-mass exemplars) — *format-blind* —
so the taxonomy is not fitted to the comparison we later want to make.


## Soft topics (MPPCA)

Topic $\sigma^2$ is fixed for the run (default `DEFAULT_TOPIC_SIGMA2`). Soft
responsibilities are checkpointed as the geometry; MAP labels are derived when
a discrete interface is required.


## PCA as a viewing device

The scatter below projects embeddings onto the first two principal components.
Classical PCA is a *view*, not the generative model. Soft lens memberships and
document EMD on soft histograms live in notebook 08.


In [ ]:
from IPython.display import display
import numpy as np
from multilogue.access import load
from multilogue.dummy import FakeClient
from multilogue.extract import make_extract_work, phrase_table
from multilogue.llm import StageRunner
from multilogue.cluster import (
    Embedder, make_label_work, make_lens_work, resolve_lenses,
)
from multilogue.mppca import (
    DEFAULT_TOPIC_SIGMA2,
    mppca_fit,
    responsibilities_to_labels,
    example_phrases_for_soft_topics,
)
from multilogue import viz


In [ ]:
units = load("synthetic", n_docs_per_group=3, units_per_doc=3)["units"]
units_by_id = {u["unit_id"]: u for u in units}
client = FakeClient()
root = FIG.parent
runner = StageRunner(root / "ckpt_cl", root / "logs_cl", max_workers=2, sleep=lambda _: None)
extraction = {}
for kind in ("concern", "benefit"):
    work = make_extract_work(
        client, kind=kind, system="e",
        user_for_unit=lambda u, k=kind: u["text"], model="fake-small",
        sentinel=f"NO_{kind.upper()}",
    )
    extraction[kind] = runner.run(
        f"ex_{kind}", units, lambda u: u["unit_id"], work, f"ex|{kind}", quiet=True,
    )
phrases = phrase_table(
    extraction, units_by_id, kinds=["concern", "benefit"], extra_unit_fields=["group"],
)
k = 4
embedder = Embedder(root / "emb", client, model="fake-emb")
E = np.asarray(embedder(phrases.phrase.tolist()), float)
fit = mppca_fit(E, k=k, sigma2=DEFAULT_TOPIC_SIGMA2, seed=0)
R = fit["responsibilities"]
means = fit["means"]
labels = responsibilities_to_labels(R)
phrases = phrases.copy()
phrases["cluster"] = labels  # harden view for bars / lens map only
# Cosine similarity to MAP topic mean (diagnostic)
valid = labels >= 0
phrases["centroid_sim"] = 0.0
phrases.loc[valid, "centroid_sim"] = (E[valid] * means[labels[valid]]).sum(1)
print(f"topic σ²={fit['sigma2']}; EM iters={fit['n_iter']}")
print(f"mean max responsibility: {R.max(axis=1).mean():.3f}")
(root / "clusters").mkdir(parents=True, exist_ok=True)
np.save(root / "clusters" / "responsibilities.npy", R)
np.save(root / "clusters" / "topic_means.npy", means)


In [ ]:
viz.cluster_size_bars(phrases, path=FIG / "03_cluster_sizes.png")
viz.embedding_scatter(E, phrases["cluster"], path=FIG / "03_pca_clusters.png")


Cluster-size bars use the MAP harden view so empty topics are visible. Soft
geometry remains in $R$. Hopeless overlap in the PCA scatter suggests revisiting
$k$ or the phrase set before interpreting shares — that decision is **Assess**,
not a format-gap maximisation.


## Assess — interactive subject grain (Bishop–Tipping)

Choosing how many subjects belong in the catalogue is closer to “what is the
human grain of this corpus?” than to “which quote threshold?”. Following
Bishop & Tipping’s hierarchical visualisation (TPAMI 1998), the analyst sees
**both**:

1. **Geometry** — a latent view of the current soft fit (updates after merge /
   split; cheap to redraw).
2. **Semantics** — an LLM catalogue of the same bags (exemplars + names).

Neither view replaces the other. Merge / split / keep is human feedback — not
automatic maximisation of a study metric $\ell$. Under *assess without the
question*, this loop is Assess; lenses, format gap, and multiverse under a
**locked** catalogue are Address.

**Label-on-demand:** every merge/split re-plots immediately; LLM names refresh
only when you ask (dirty topics), so exploration is not blocked by API latency.


In [ ]:
import os
from multilogue.subject_grain_shell import SubjectGrainShell, ipywidgets_available
from multilogue.cluster import make_label_work

label_work = make_label_work(
    client, system="label",
    prompt_template="Label these:\n<<PHRASES>>", model="fake-small",
)
shell = SubjectGrainShell(
    E, fit, phrases, phrase_col="phrase", seed=0, label_work=label_work,
)
# Widgets for live try-combinations; CI / smoke force the scripted path.
use_widgets = (
    ipywidgets_available()
    and os.environ.get("MULTILOGUE_SUBJECT_GRAIN_WIDGETS", "1") != "0"
)
if use_widgets:
    print("Layer B: interactive shell (merge / split / undo / name dirty / lock)")
    shell.display()
else:
    print("Layer A scripted path (no widgets / MULTILOGUE_SUBJECT_GRAIN_WIDGETS=0)")
    display(shell.catalogue())
    shell.fit_view(path=FIG / "03_topic_fit_view.png", show=True)
    # Demo one merge then undo so smoke stays deterministic
    if shell.k >= 3:
        shell.merge([0, 1])
        print("after merge: k =", shell.k, "dirty =", sorted(shell.dirty))
        shell.fit_view(path=FIG / "03_topic_fit_after_merge.png", show=True)
        shell.name_dirty()
        display(shell.catalogue())
        shell.undo()
        print("after undo: k =", shell.k)
    lock = shell.lock(
        path=root / "clusters" / "subject_grain_lock.json",
        k_multiverse="hold",
    )
    print("locked subject grain:", lock["primary_k"], lock["k_multiverse"])
    display(lock)


Lock before Address spend: the artefact records `primary_k` and a hold /
$\pm 1$ multiverse policy. Wide $k$ grids belong in an explicit one-factor
resolution sweep — not in the default factorial multiverse (see notebook 09).


## Blind labels and lenses (harden view)

Exemplars are ranked by soft mass on each topic; the LLM never sees raw
responsibility matrices. Researchers may edit the lens mapping afterwards — the
first pass stays blind. After a subject-grain lock, labels here name the
*locked* leaves.


In [ ]:
# Re-bind fit / R from the shell if the scripted path ran; widgets leave `fit` as initial.
R = shell.responsibilities
fit = shell.fit
means = fit["means"]
k = shell.k
labels = responsibilities_to_labels(R)
phrases = phrases.copy()
phrases["cluster"] = labels
valid = labels >= 0
phrases["centroid_sim"] = 0.0
phrases.loc[valid, "centroid_sim"] = (E[valid] * means[labels[valid]]).sum(1)

cluster_units = example_phrases_for_soft_topics(
    phrases.phrase.tolist(), R, n_top=8, n_sample=2, seed=0,
)
# Drop empty components for labelling
cluster_units = [cu for cu in cluster_units if cu["examples"]]
lab = runner.run(
    "labels", cluster_units, lambda cu: cu["key"],
    make_label_work(
        client, system="label",
        prompt_template="Label these:\n<<PHRASES>>", model="fake-small",
    ),
    "labels|v1", quiet=True,
)
names = {
    cu["cluster"]: lab.get(cu["key"], {}).get("label", f"cluster {cu['cluster']}")
    for cu in cluster_units
}
print(names)
cl_desc = "\n".join(
    f"{c}: {names.get(c, f'topic {c}')} — (mass {R[:, c].sum():.1f})" for c in range(k)
)
lens_runs = runner.run(
    "lenses", [1, 2], lambda r: str(r),
    make_lens_work(
        client, system="lens", prompt_template="Group:\n<<CLUSTERS>>",
        cluster_description=cl_desc, model="fake-small", k=k,
    ),
    f"lenses|{cl_desc[:60]}", quiet=True,
)
mapping, edited = resolve_lenses(k, lens_runs)
phrases["lens"] = phrases["cluster"].map(mapping)
print("lens mapping:", mapping, "edited:", edited)


In [ ]:
viz.embedding_scatter(
    E, phrases["lens"],
    title="Embedding space coloured by lens (PCA view)",
    path=FIG / "03_pca_lenses.png",
)


# Outlook

Soft responsibilities $R$ and topic means feed notebook 08 (soft histograms →
document EMD; soft lenses under a separate fixed $\sigma^2_{\mathrm{lens}}$).
Notebook 04 can still use MAP or lens columns for document-weighted shares; soft
composition is the CIP-0017 default for transport. Notebook 09 consumes a
**locked** subject grain under format-gap $\ell$ (multiverse holds $k$).
